# Brian: Source-to-Notes Prototype v0.1

## Purpose

This notebook takes one clean research PDF, extracts page-level text, creates 3–5 claim-level research notes, validates the page number and exact evidence snippet, saves the notes as CSV, and reads the saved file back into pandas.

**Workflow:** PDF → page text → structured AI notes → automatic evidence check → human review → CSV → verification


## Setup

Run this cell once if the required packages are not already installed:

```python
%pip install -U openai python-dotenv pydantic pandas pypdf
```

Create a `.env` file in the project root folder containing:

```text
OPENAI_API_KEY=your_api_key_here
```

Do not place the API key directly in the notebook or upload the `.env` file to GitHub.


In [ ]:
import os
import sys
from pathlib import Path
from typing import Literal

import pandas as pd
import pypdf
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY was not found. Add it to a .env file in the project root."
    )

client = OpenAI(api_key=api_key)

print("Setup complete. API key loaded securely.")
print("Python version:", sys.version.split()[0])
print("pandas version:", pd.__version__)


In [ ]:
response = client.responses.create(
    model="gpt-5-mini",
    input="Reply with exactly this text: API connection successful."
)

print(response.output_text)


## Input PDF

Place the research PDF in the same folder as this notebook and update the filename below.


In [ ]:
PDF_FILE = Path("research_paper.pdf")
OUTPUT_FILE = Path("source_to_notes_v0_1.csv")

if not PDF_FILE.exists():
    raise FileNotFoundError(
        f"Could not find {PDF_FILE}. Put the PDF in the notebook folder "
        "or update PDF_FILE to the correct path."
    )

print("Input PDF:", PDF_FILE.resolve())
print("Output CSV:", OUTPUT_FILE.resolve())


## Extract page-level text

In [ ]:
reader = pypdf.PdfReader(str(PDF_FILE))

pages = []
for page_number, page in enumerate(reader.pages, start=1):
    pages.append({
        "page_number": page_number,
        "page_text": (page.extract_text() or "").strip()
    })

pages_df = pd.DataFrame(pages)
print(f"Extracted {len(pages_df)} pages.")
display(pages_df.head())


In [ ]:
empty_pages = pages_df["page_text"].str.strip().eq("").sum()
print("Pages with no extracted text:", empty_pages)

if empty_pages > 0:
    print("Some pages may require OCR. Affected notes should remain Needs Review.")


## Structured output schema

`claim_summary` is a summary or paraphrase.

`exact_evidence_snippet` must be copied exactly from the PDF and stored separately.


In [ ]:
class ResearchNote(BaseModel):
    source_page_number: int | None = Field(
        description="PDF page number, or null when it cannot be verified."
    )
    claim_summary: str = Field(
        description="Summary or paraphrase without quotation marks."
    )
    exact_evidence_snippet: str = Field(
        description="Exact copied source text, or Needs Review."
    )
    status: Literal["Supported", "Needs Review"]


class ResearchNotesOutput(BaseModel):
    notes: list[ResearchNote]


## Prepare page-labeled source text

In [ ]:
page_blocks = [
    f"--- PDF PAGE {row['page_number']} ---\n{row['page_text']}"
    for row in pages
]
paper_text = "\n\n".join(page_blocks)
print("Characters sent to model:", len(paper_text))


## Generate 3–5 claim-level notes

In [ ]:
instructions = '''
You are a Source-to-Notes research assistant.

Create 3 to 5 claim-level research notes from the supplied research paper.

Rules:
1. Each note contains one clear claim or summarized idea.
2. claim_summary is a paraphrase and must not be placed in quotation marks.
3. exact_evidence_snippet must be copied exactly from the supplied page text.
4. source_page_number must match the page label containing the evidence.
5. If evidence or page number cannot be verified, use:
   source_page_number = null
   exact_evidence_snippet = Needs Review
   status = Needs Review
6. Do not invent quotations, page numbers, findings, or interpretations.
7. Use Supported only when page number and exact evidence are verifiable.
'''

response = client.responses.parse(
    model="gpt-5-mini",
    input=[
        {"role": "system", "content": instructions},
        {"role": "user", "content": paper_text},
    ],
    text_format=ResearchNotesOutput,
)

structured_output = response.output_parsed

if structured_output is None:
    raise ValueError("No parsed structured output was returned.")

print(f"Model produced {len(structured_output.notes)} notes.")


## Convert notes to a DataFrame

In [ ]:
records = []

for index, note in enumerate(structured_output.notes, start=1):
    records.append({
        "record_id": f"BRIAN-S2N-{index:03d}",
        "source_filename": PDF_FILE.name,
        "source_page_number": note.source_page_number,
        "claim_summary": note.claim_summary,
        "exact_evidence_snippet": note.exact_evidence_snippet,
        "status": note.status,
        "reviewer_decision": "Pending",
        "reviewer_comment": ""
    })

notes_df = pd.DataFrame(records)
display(notes_df)


## Automatic evidence validation

This step checks whether the evidence appears exactly on the stated PDF page.


In [ ]:
page_text_lookup = {
    row["page_number"]: row["page_text"]
    for row in pages
}

def validate_note(row):
    page_number = row["source_page_number"]
    evidence = str(row["exact_evidence_snippet"]).strip()

    if pd.isna(page_number):
        return "Needs Review", "Page number is missing."

    try:
        page_number = int(page_number)
    except (TypeError, ValueError):
        return "Needs Review", "Page number is invalid."

    if page_number not in page_text_lookup:
        return "Needs Review", "Page number is outside the PDF range."

    if evidence in ("", "Needs Review"):
        return "Needs Review", "Exact evidence was not verified."

    if evidence not in page_text_lookup[page_number]:
        return "Needs Review", "Evidence was not found exactly on the stated page."

    return "Supported", "Exact evidence was found on the stated page."

validation_results = notes_df.apply(validate_note, axis=1)
notes_df["status"] = [result[0] for result in validation_results]
notes_df["automatic_check_comment"] = [result[1] for result in validation_results]

display(notes_df)


## Human review

Manually compare every note with the PDF. Change `reviewer_decision` to `Approved`, `Corrected`, or `Rejected`, and add a comment.


In [ ]:
# Example:
# notes_df.loc[0, "reviewer_decision"] = "Approved"
# notes_df.loc[0, "reviewer_comment"] = "Claim and evidence verified manually."

display(notes_df)


## Save structured notes as CSV

In [ ]:
notes_df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved structured notes to: {OUTPUT_FILE.resolve()}")


## Read the saved CSV back into pandas and verify

In [ ]:
verified_df = pd.read_csv(OUTPUT_FILE)
display(verified_df)

required_columns = [
    "record_id",
    "source_filename",
    "source_page_number",
    "claim_summary",
    "exact_evidence_snippet",
    "status",
    "reviewer_decision",
    "reviewer_comment",
    "automatic_check_comment"
]

missing_columns = [
    column for column in required_columns
    if column not in verified_df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

if not 3 <= len(verified_df) <= 5:
    raise ValueError(f"Expected 3 to 5 notes, found {len(verified_df)}.")

if verified_df["record_id"].duplicated().any():
    raise ValueError("Record IDs are not unique.")

pd.testing.assert_frame_equal(
    notes_df.reset_index(drop=True),
    verified_df.reset_index(drop=True),
    check_dtype=False
)

print("Verification passed.")


## Run Log

| Item | Value |
|---|---|
| Input filename | Add after run |
| Output filename | source_to_notes_v0_1.csv |
| Python version | Printed above |
| pandas version | Printed above |
| PDF pages extracted | Printed above |
| Number of notes | Printed above |
| Automatic verification | Passed / Failed |
| Human review status | Pending / Complete |
| Problems encountered | Add notes here |
| Date run | Add date here |
